# 第05章 几何变换（跟随代码）

> 《OpenCV轻松入门：面向Python》（李立宗）第 5 章跟随代码。**全量收录原书代码（例 5.1～例 5.20）**，缺失例题已按书补全，代码以网页笔记 docs/opencv/ch05.md 为准（含笔误修正与参数修正）。
> cv2.imshow / cv_show 需在本地 Python 环境运行（弹窗显示，Notebook 服务内看不到窗口）；运行前请确认**工作目录为素材根目录**（本文件夹，picture*.jpg 等素材都在这里），否则 imread 返回 None。
> 素材替换：原书 lena.bmp、test.bmp、demo.bmp 统一用 picture1.jpg 顶替，代码逻辑一字未改；个别坐标在非方图上会越界（例 5.8、例 5.20，见对应说明）。remap 系列例题原书用的 `itemset` 已被 NumPy 2.0 移除，笔记与本书已改为等价的 `mapx[i, j] = 值` 写法。

## 本章总览

几何变换（geometric transform）是指将一幅图像映射到另外一幅图像内的操作。全章例题：例 5.1～例 5.3（缩放）、例 5.4（翻转）、例 5.5～例 5.7（仿射：平移/旋转/三点仿射）、例 5.8（透视）、例 5.9～例 5.20（重映射），另有 5.6 节极坐标变换（拓展）。

- **缩放** `cv2.resize(src, dsize, fx, fy, interpolation)`：dsize 是（宽, 高），与 shape 的（行, 列）相反；不给 dsize 时按 fx、fy 比例算。缩小首选 INTER_AREA，放大用 INTER_CUBIC 或 INTER_LINEAR；
- **翻转** `cv2.flip(src, flipCode)`：0 绕 x 轴上下翻、正数绕 y 轴左右翻、负数双向翻，与切片反转完全等价；
- **仿射** `cv2.warpAffine(src, M, dsize, flags, borderMode, borderValue)`：一个 2×3 矩阵 M 定变换——平移把 tx、ty 放 M 第三列；旋转用 `cv2.getRotationMatrix2D(center, angle, scale)` 生成矩阵；任意仿射用 `cv2.getAffineTransform(src, dst)` 三点定矩阵。直线仍直、平行线仍平行；
- **透视** `cv2.warpPerspective` + `cv2.getPerspectiveTransform`：3×3 矩阵、四对点定矩阵，矩形可映成任意四边形；
- **重映射** `cv2.remap(src, map1, map2, interpolation)`：mapx 存列号、mapy 存行号，反向取样，浮点坐标靠插值落位，越界点统一补 0；复制、翻转、转置都是特例；
- **极坐标（拓展）** `cv2.warpPolar`：把圆形区域展开成矩形条，宽度是半径轴、高度是角度轴。

In [ ]:
import cv2

# cv_show 辅助函数：原书跟随代码沿用的显示封装（本地运行显示窗口用，弹窗显示后按任意键关闭）
def cv_show(name, img):
    cv2.imshow(name, img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


## 5.1 缩放

OpenCV 用 `cv2.resize()` 实现缩放，语法 `dst = cv2.resize( src, dsize[, fx[, fy[, interpolation]]] )`。目标图像的大小有两种指定方式，二者取其一：

- **通过参数 dsize 指定**：第 1 个值是宽度（列数 cols，与 fx 相关），第 2 个值是高度（行数 rows，与 fy 相关）——**与 shape 属性的（行, 列）顺序正好相反**；
- **通过参数 fx 和 fy 指定**：dsize 为 None 时按比例计算，dsize = Size(round(fx·cols), round(fy·rows))。

interpolation（插值）解决"映射落空"的赋值问题——放大图像必然多出无法直接映射到原图的像素，反向映射也可能把目标像素映射到原图的非整数位置上。常用插值：`cv2.INTER_NEAREST` 最邻近、`cv2.INTER_LINEAR` 双线性（默认）、`cv2.INTER_CUBIC` 三次样条（4×4 近邻拟合）、`cv2.INTER_AREA` 区域插值、`cv2.INTER_LANCZOS4`（8×8 近邻）。经验：**缩小图像用区域插值（INTER_AREA）效果最好；放大图像用三次样条（INTER_CUBIC）或双线性（INTER_LINEAR）都能取得较好效果**——三次样条较慢，双线性快且效果并不逊色，所以它才是默认值。本节包含例 5.1～例 5.3。

### 例 5.1 · 对数组做等大小缩放，暴露 dsize 的顺序问题

把 2 行 4 列数组的 `shape[:2]` 直接当 dsize 传入 `cv2.resize`，结果得到 4 行 2 列——dsize 把第 1 个值当**列数**用。结论：shape 第 1 个值是行数，dsize 第 1 个值是列数（宽）。等大小的方图上这个坑完全隐形。

In [ ]:
# ===== 5.1 例 5.1：dsize 传 shape[:2]，结果不是等大小 =====
import cv2
import numpy as np

img = np.ones([2, 4, 3], dtype=np.uint8)   # 2 行 4 列 3 通道的全 1 数组
size = img.shape[:2]                        # (2, 4)：shape 里是（行, 列）
rst = cv2.resize(img, size)                 # dsize 却把第 1 个值当列数用
print("img.shape=\n", img.shape)
print("rst.shape=\n", rst.shape)

### 例 5.2 · 用 dsize 指定大小，横向缩到 0.9 倍、纵向缩到 0.5 倍

`size = (int(cols*0.9), int(rows*0.5))`——注意 size 里写的是（列数×0.9, 行数×0.5），返回 shape 却是（行数×0.5, 列数×0.9），正是例 5.1 说的顺序差。

In [ ]:
# ===== 5.1 例 5.2：dsize =（新宽, 新高） =====
import cv2

img = cv2.imread("picture1.jpg")            # 原书 test.bmp
rows, cols = img.shape[:2]                  # rows=1080, cols=1920
size = (int(cols*0.9), int(rows*0.5))       # （列数×0.9, 行数×0.5）
rst = cv2.resize(img, size)
print("img.shape=", img.shape)
print("rst.shape=", rst.shape)

### 例 5.3 · 用 fx、fy 指定比例

`cv2.resize(img, None, fx=2, fy=0.5)`：fx=2 是水平方向缩放（列数 1920×2=3840），fy=0.5 是垂直方向缩放（行数 1080×0.5=540），dsize 位置必须给 None。

In [ ]:
# ===== 5.1 例 5.3：fx 管列、fy 管行 =====
import cv2

img = cv2.imread("picture1.jpg")            # 原书 test.bmp
rst = cv2.resize(img, None, fx=2, fy=0.5)   # dsize 必须给 None
print("img.shape=", img.shape)
print("rst.shape=", rst.shape)

**补充演示**（跟随代码原有）：`picture3.jpg` 的两种缩放方式——`dsize=(640, 480)` 直接指定大小，与 `fx=0.5, fy=0.5` 按比例缩放。

In [210]:
#缩放
import cv2
import numpy as np

img=cv2.imread("picture3.jpg")
#====通过参数dsize来设置缩放大小====
dst=cv2.resize(img,(640,480))
cv_show("original",img)
cv_show("dst",dst)
##====通过参数fx和fy来设置缩放比例====
dst=cv2.resize(img,None,fx=0.5,fy=0.5)
cv_show("original",img)
cv_show("dst",dst)

## 5.2 翻转

图像翻转（flip）用 `cv2.flip( src, flipCode )` 实现：flipCode=0 绕 x 轴翻转（上下翻转）、正数绕 y 轴翻转（左右翻转）、负数绕 x 轴 y 轴同时翻转。翻转没有任何"黑科技"，就是数组反转——`cv2.flip(g, 0)` 等价 `g[::-1, :]`、`cv2.flip(g, 1)` 等价 `g[:, ::-1]`、`cv2.flip(g, -1)` 等价 `g[::-1, ::-1]`；同一符号类别的效果完全一样（flipCode=2 与 1 相同、-3 与 -1 相同）。

### 例 5.4 · 三种翻转一次看全

对 `picture1.jpg`（原书 lena.bmp）分别取 flipCode=0、1、-1：四张图对照，(b) 头朝下，(c) 左右镜像，(d) 是 (b)、(c) 的叠加。

In [ ]:
# ===== 5.2 例 5.4：flipCode 取 0、1、−1 =====
import cv2

img = cv2.imread("picture1.jpg")    # 原书 lena.bmp
x = cv2.flip(img, 0)                # 绕 x 轴：上下翻转
y = cv2.flip(img, 1)                # 绕 y 轴：左右翻转
xy = cv2.flip(img, -1)              # 双向：上下 + 左右

cv2.imshow("img", img)
cv2.imshow("x", x)
cv2.imshow("y", y)
cv2.imshow("xy", xy)
cv2.waitKey()
cv2.destroyAllWindows()

## 5.3 仿射

仿射变换（affine transform）指图像通过一系列几何变换实现平移、旋转等多种操作，保持两条性质：**平直性**（变换后直线仍然是直线）与**平行性**（变换后平行线仍然是平行线）。函数 `cv2.warpAffine( src, M, dsize[, flags[, borderMode[, borderValue]]] )` 靠一个 2×3 的变换矩阵（映射矩阵）M 完成变换——做哪种仿射变换，完全取决于矩阵 M：

- **M**：2×3 变换矩阵，不同的 M 实现不同的仿射变换；
- **dsize**：输出图像的尺寸大小；
- **flags**：插值方法，默认 INTER_LINEAR；取 WARP_INVERSE_MAP 时 M 直接作为逆变换使用（从 dst 反查 src）；
- **borderMode**：边界模式，默认 BORDER_CONSTANT；
- **borderValue**：边界填充值，默认 0。

忽略可选参数后就是三件事：图、矩阵、画布大小。本节包含例 5.5（平移）、例 5.6（旋转）、例 5.7（更复杂的仿射）。

### 5.3.1 平移

### 例 5.5 · 自定义转换矩阵完成平移

把图像向右移 100、向下移 200：M 的第 3 列放平移量，`M = np.float32([[1, 0, x], [0, 1, y]])`，前两列保持原样（不旋转、不缩放）。dsize 仍是（宽, 高）；平移出画布的部分丢弃，空出的位置填 borderValue（默认黑）。记忆口径：**第三列正数 = 内容往右、往下跑**（一旦加了 WARP_INVERSE_MAP 标志，公式直接用 M 本身，同样的 M 方向全反）。

In [ ]:
# ===== 5.3.1 例 5.5：M 第三列 =（右移量, 下移量） =====
import cv2
import numpy as np

img = cv2.imread("picture1.jpg")            # 原书 lena.bmp
height, width = img.shape[:2]
x = 100                                     # 右移 100
y = 200                                     # 下移 200
M = np.float32([[1, 0, x], [0, 1, y]])      # 2×3 平移矩阵
move = cv2.warpAffine(img, M, (width, height))  # dsize 仍是（宽, 高）

cv2.imshow("original", img)
cv2.imshow("move", move)
cv2.waitKey()
cv2.destroyAllWindows()

### 5.3.2 旋转

### 例 5.6 · 图像旋转

`cv2.getRotationMatrix2D( center, angle, scale )` 直接生成旋转矩阵：center 是 (x, y) 旋转中心（应为（宽/2, 高/2）——原书正文写成（高/2, 宽/2）属笔误，例 5.6 的代码是对的）；angle 旋转角度，正数逆时针、负数顺时针；scale 变换尺度。生成后交给 `warpAffine` 执行，画布仍是原图大小，转出画布的四角内容变黑。

想旋转 45° 不裁边（补充技巧，非原书内容）：把 dsize 扩到能装下整个旋转外接矩形并补偿平移列——`nw = int(h*sin + w*cos)`、`M[0, 2] += nw/2 - w/2`、`M[1, 2] += nh/2 - h/2`。

In [ ]:
# ===== 5.3.2 例 5.6：中心旋转 45°，缩至 0.6 =====
import cv2

img = cv2.imread("picture1.jpg")                       # 原书 lena.bmp
height, width = img.shape[:2]
M = cv2.getRotationMatrix2D((width/2, height/2), 45, 0.6)
rotate = cv2.warpAffine(img, M, (width, height))

cv2.imshow("original", img)
cv2.imshow("rotation", rotate)
cv2.waitKey()
cv2.destroyAllWindows()

### 5.3.3 更复杂的仿射变换

### 例 5.7 · 三点法仿射

`cv2.getAffineTransform( src, dst )` 由**三对点**（平行四边形的左上角、右上角、左下角）确定 2×3 矩阵（6 个未知数、6 个方程），其余点的映射关系按这三点的关系推算。想设计"斜切"效果，改 p2 三个点即可，不必自己推矩阵。验证矩阵的办法：把 M 作用到 p1 各点的齐次坐标，逐一等于 p2 的对应点。

In [ ]:
# ===== 5.3.3 例 5.7：三个点对生成任意仿射 =====
import cv2
import numpy as np

img = cv2.imread("picture1.jpg")            # 原书 lena.bmp
rows, cols, ch = img.shape
p1 = np.float32([[0, 0], [cols-1, 0], [0, rows-1]])                    # 原图三个顶点
p2 = np.float32([[0, rows*0.33], [cols*0.85, rows*0.25],
                 [cols*0.15, rows*0.7]])                               # 目标三个顶点
M = cv2.getAffineTransform(p1, p2)          # 2×3 矩阵
dst = cv2.warpAffine(img, M, (cols, rows))

cv2.imshow("original", img)
cv2.imshow("result", dst)
cv2.waitKey()
cv2.destroyAllWindows()

**补充演示**（跟随代码原有）：仿射三合一——平移（`borderValue=(255,255,255)` 白色填充，并演示 WARP_INVERSE_MAP 下同一矩阵方向全反）、中心旋转 45°（scale=0.7）、三点法仿射（p2 取点与例 5.7 略有不同）。

In [212]:
#仿射


#===平移===#
img = cv2.imread("picture3.jpg")
h,w=img.shape[:2]
x=100
y=50
M=np.float32([[1,0,x],[0,1,y]])

move=cv2.warpAffine(img,M,(w,h),borderValue=(255,255,255))
cv_show("move",move)
move_reflect=cv2.warpAffine(img,M,(w,h),flags=cv2.WARP_INVERSE_MAP,borderValue=(255,255,255))
cv_show("move_reflect",move_reflect)

#===旋转===#

#获取转换矩阵
retal=cv2.getRotationMatrix2D((w/2,h/2),45,0.7)#center:旋转中心,angle:旋转角度,scale:缩放因子
#进行旋转
rotate=cv2.warpAffine(img,retal,(w,h))
cv_show("rotate",rotate)


#===更复杂的仿射变换===#

rows,cols,ch=img.shape

p1=np.float32([[0,0],[cols-1,0],[0,rows-1]])
p2=np.float32([[0,rows*0.3],[cols*0.8,rows*0.2],[cols*0.1,rows*0.7]])

M=cv2.getAffineTransform(p1,p2)#输入前后两组三个点的坐标，输出仿射变换矩阵
transform=cv2.warpAffine(img,M,(cols,rows))
cv_show("transform",transform)

## 5.4 透视

仿射能把矩形映成任意**平行四边形**；透视变换（perspective transform）更进一步，能把矩形映成**任意四边形**——平行性不再保证，这正是拍照时"近大远小"的几何模型。`cv2.warpPerspective( src, M, dsize[, flags[, borderMode[, borderValue]]] )` 用 **3×3** 变换矩阵；矩阵由 `cv2.getPerspectiveTransform( src, dst )` 从**四对点**生成（注意是四个点，与仿射的三个点不同）。透视多一步"除法"，要靠齐次坐标实现——3×3 矩阵有 9 个元素、8 个自由度，所以恰好需要四对点。

### 例 5.8 · 把平行四边形矫正成矩形

原图四边形顶点 pts1 映射到目标矩形顶点 pts2：`getPerspectiveTransform` 求出 3×3 矩阵后交给 `warpPerspective`。

> **原书笔误**：`pts2` 把 rows 写进了 x 坐标、cols 写进了 y 坐标（x 该看列数 cols、y 该看行数 rows）。原书 demo.bmp 恰好是方图碰巧看不出问题；换成 1080×1920 的真图后，目标矩形下边 y 最大到 cols-50=1870，越过得 1080 行的画布底部，内容被裁。稳妥写法：`pts2 = np.float32([[50, 50], [cols-50, 50], [50, rows-50], [cols-50, rows-50]])`。下面代码保留原书写法以便对照。

In [ ]:
# ===== 5.4 例 5.8：四点对生成 3×3 透视矩阵 =====
import cv2
import numpy as np

img = cv2.imread("picture1.jpg")            # 原书 demo.bmp
rows, cols = img.shape[:2]
print(rows, cols)
pts1 = np.float32([[150, 50], [400, 50], [60, 450], [310, 450]])   # 原图四边形顶点
pts2 = np.float32([[50, 50], [rows-50, 50],
                   [50, cols-50], [rows-50, cols-50]])              # 目标矩形顶点（原书写法）
M = cv2.getPerspectiveTransform(pts1, pts2)     # 3×3 矩阵
dst = cv2.warpPerspective(img, M, (cols, rows))

cv2.imshow("img", img)
cv2.imshow("dst", dst)
cv2.waitKey()
cv2.destroyAllWindows()

## 5.5 重映射

重映射（remap）把一幅图像内的像素点放置到另一幅图像内的指定位置，`cv2.remap( src, map1, map2, interpolation[, borderMode[, borderValue]] )` 提供自定义接口。核心是**反向映射**（inverse mapping）：新图像的每个像素去原图的哪里取值？map1 针对 x 坐标（mapx）、map2 针对 y 坐标（mapy）：

- **mapx 存的是列号，mapy 存的是行号**——想把目标图 A 点映为原图第 0 行第 3 列的像素 B，就把 A 处的 mapx 设为 3、mapy 设为 0；
- mapx、mapy 都是**浮点数**，所以目标可以"反向映射"到原图两个像素之间的位置（这个位置本没有像素值），由 interpolation 参数插值定值（不支持 INTER_AREA）；
- 映射到原图外的位置统一补 borderValue（默认 0）；
- 缩放、翻转、畸变都只是它的特例。

### 5.5.1 映射参数的理解

### 例 5.9 · 把目标数组的所有点都映射到原图同一个点

mapx 全 3、mapy 全 0：目标图每个位置都取原图第 0 行第 3 列的像素——整幅图变成同一个值。

In [ ]:
# ===== 5.5.1 例 5.9：常量映射表 =====
import cv2
import numpy as np

img = np.random.randint(0, 256, size=[4, 5], dtype=np.uint8)   # 4 行 5 列随机数组
rows, cols = img.shape
mapx = np.ones(img.shape, np.float32) * 3    # 所有位置的列号都是 3
mapy = np.ones(img.shape, np.float32) * 0    # 所有位置的行号都是 0
rst = cv2.remap(img, mapx, mapy, cv2.INTER_LINEAR)
print("img=\n", img)
print("rst=\n", rst)

### 5.5.2 复制

### 例 5.10 · 数组复制

mapx 取各位置的**列号** j、mapy 取各位置的**行号** i，映射就是"原样照搬"，rst 与 img 完全相等。

> **itemset 已不存在**：本节起原书用 `mapx.itemset((i, j), 值)` 写映射表，NumPy 2.0 起该方法已删除（实测 numpy 2.2.6 直接 AttributeError）。等价写法是 `mapx[i, j] = 值`，下面例题按新写法给出，变量名 mapx、mapy 沿用原书。映射表也不必逐点循环：`mapx = np.tile(np.arange(cols, dtype=np.float32), (rows, 1))`、`mapy = np.repeat(np.arange(rows, dtype=np.float32)[:, None], cols, axis=1)` 一行生成，实测与循环版逐像素相等。

In [ ]:
# ===== 5.5.2 例 5.10：列号表 + 行号表 = 复制 =====
import cv2
import numpy as np

img = np.random.randint(0, 256, size=[4, 5], dtype=np.uint8)
rows, cols = img.shape
mapx = np.zeros(img.shape, np.float32)
mapy = np.zeros(img.shape, np.float32)
for i in range(rows):
    for j in range(cols):
        mapx[i, j] = j                  # 原书：mapx.itemset((i,j), j)——列号
        mapy[i, j] = i                  # 原书：mapy.itemset((i,j), i)——行号
rst = cv2.remap(img, mapx, mapy, cv2.INTER_LINEAR)
print("rst=\n", rst)

### 例 5.11 · 图像复制

真实图像上的复制映射：对 `picture1.jpg`（原书 lena.bmp）的 mapx、mapy 逐点填列号、行号，remap 结果与原图逐像素一致。

In [ ]:
# ===== 5.5.2 例 5.11：真实图像上的复制映射 =====
import cv2
import numpy as np

img = cv2.imread("picture1.jpg")        # 原书 lena.bmp
rows, cols = img.shape[:2]
mapx = np.zeros(img.shape[:2], np.float32)
mapy = np.zeros(img.shape[:2], np.float32)
for i in range(rows):
    for j in range(cols):
        mapx[i, j] = j                  # 列号表
        mapy[i, j] = i                  # 行号表
rst = cv2.remap(img, mapx, mapy, cv2.INTER_LINEAR)

cv2.imshow("original", img)
cv2.imshow("result", rst)
cv2.waitKey()
cv2.destroyAllWindows()

### 5.5.3 绕 x 轴翻转

### 例 5.12 · 数组绕 x 轴翻转

x 坐标（列）不变，y 坐标（行）以 x 轴为对称轴交换：mapx 不变（仍是列号），mapy 调整为"总行数 − 1 − 当前行号"——行号从 0 起，当前行号 + 对称行号 = 总行数 − 1。4 行数组的 mapy 依次是 3、2、1、0，实测 rst 与 `img[::-1, :]` 完全相等。

In [ ]:
# ===== 5.5.3 例 5.12：mapy 行号倒置 =====
import cv2
import numpy as np

img = np.random.randint(0, 256, size=[4, 5], dtype=np.uint8)
rows, cols = img.shape
mapx = np.zeros(img.shape, np.float32)
mapy = np.zeros(img.shape, np.float32)
for i in range(rows):
    for j in range(cols):
        mapx[i, j] = j                          # 列号不变
        mapy[i, j] = rows - 1 - i               # 原书：mapy.itemset((i,j), rows-1-i)
rst = cv2.remap(img, mapx, mapy, cv2.INTER_LINEAR)
print("rst=\n", rst)

### 例 5.13 · 图像绕 x 轴翻转

真实图像上的绕 x 轴翻转，与 `cv2.flip(img, 0)` 等效。

In [ ]:
# ===== 5.5.3 例 5.13：图像绕 x 轴翻转 =====
import cv2
import numpy as np

img = cv2.imread("picture1.jpg")        # 原书 lena.bmp
rows, cols = img.shape[:2]
mapx = np.zeros(img.shape[:2], np.float32)
mapy = np.zeros(img.shape[:2], np.float32)
for i in range(rows):
    for j in range(cols):
        mapx[i, j] = j
        mapy[i, j] = rows - 1 - i
rst = cv2.remap(img, mapx, mapy, cv2.INTER_LINEAR)

cv2.imshow("original", img)
cv2.imshow("result", rst)
cv2.waitKey()
cv2.destroyAllWindows()

### 5.5.4 绕 y 轴翻转

### 例 5.14 · 数组绕 y 轴翻转

y 坐标（行）不变，x 坐标（列）以 y 轴为对称轴交换：mapy 不变（仍是行号），mapx 调整为"总列数 − 1 − 当前列号"。mapx 每行都是 4、3、2、1、0，实测 rst 与 `img[:, ::-1]` 完全相等。

In [ ]:
# ===== 5.5.4 例 5.14：mapx 列号倒置 =====
import cv2
import numpy as np

img = np.random.randint(0, 256, size=[4, 5], dtype=np.uint8)
rows, cols = img.shape
mapx = np.zeros(img.shape, np.float32)
mapy = np.zeros(img.shape, np.float32)
for i in range(rows):
    for j in range(cols):
        mapx[i, j] = cols - 1 - j               # 原书：mapx.itemset((i,j), cols-1-j)
        mapy[i, j] = i                          # 行号不变
rst = cv2.remap(img, mapx, mapy, cv2.INTER_LINEAR)
print("rst=\n", rst)

### 例 5.15 · 图像绕 y 轴翻转

真实图像上的绕 y 轴翻转，与 `cv2.flip(img, 1)` 等效。

In [ ]:
# ===== 5.5.4 例 5.15：图像绕 y 轴翻转 =====
import cv2
import numpy as np

img = cv2.imread("picture1.jpg")        # 原书 lena.bmp
rows, cols = img.shape[:2]
mapx = np.zeros(img.shape[:2], np.float32)
mapy = np.zeros(img.shape[:2], np.float32)
for i in range(rows):
    for j in range(cols):
        mapx[i, j] = cols - 1 - j
        mapy[i, j] = i
rst = cv2.remap(img, mapx, mapy, cv2.INTER_LINEAR)

cv2.imshow("original", img)
cv2.imshow("result", rst)
cv2.waitKey()
cv2.destroyAllWindows()

### 5.5.5 绕 x 轴、y 轴翻转

### 例 5.16 · 数组绕 x 轴、y 轴翻转

两个轴一起翻：x 坐标以 y 轴为对称交换，y 坐标以 x 轴为对称交换——mapx 调整为"总列数 − 1 − 当前列号"、mapy 调整为"总行数 − 1 − 当前行号"。实测 rst 与 `img[::-1, ::-1]` 完全相等，即 `cv2.flip(img, -1)` 的效果。

In [ ]:
# ===== 5.5.5 例 5.16：两张表同时倒置 =====
import cv2
import numpy as np

img = np.random.randint(0, 256, size=[4, 5], dtype=np.uint8)
rows, cols = img.shape
mapx = np.zeros(img.shape, np.float32)
mapy = np.zeros(img.shape, np.float32)
for i in range(rows):
    for j in range(cols):
        mapx[i, j] = cols - 1 - j
        mapy[i, j] = rows - 1 - i
rst = cv2.remap(img, mapx, mapy, cv2.INTER_LINEAR)
print("rst=\n", rst)

### 例 5.17 · 图像绕 x 轴、y 轴翻转

真实图像上的双向翻转。

In [ ]:
# ===== 5.5.5 例 5.17：图像双向翻转 =====
import cv2
import numpy as np

img = cv2.imread("picture1.jpg")        # 原书 lena.bmp
rows, cols = img.shape[:2]
mapx = np.zeros(img.shape[:2], np.float32)
mapy = np.zeros(img.shape[:2], np.float32)
for i in range(rows):
    for j in range(cols):
        mapx[i, j] = cols - 1 - j
        mapy[i, j] = rows - 1 - i
rst = cv2.remap(img, mapx, mapy, cv2.INTER_LINEAR)

cv2.imshow("original", img)
cv2.imshow("result", rst)
cv2.waitKey()
cv2.destroyAllWindows()

### 5.5.6 x 轴、y 轴互换

### 例 5.18 · 数组 x 轴、y 轴互换（转置）

每个点的 x、y 坐标互换即转置：mapx 填所在位置的**行号**、mapy 填**列号**。行数和列数不一致时（本例故意用 4 行 6 列），部分位置无法映射——比如 4 行 6 列的图转置后需要 6 行，画布只有 4 行，默认情况下这些位置处理为 0：右侧两列全黑，左侧 4×4 恰是原图的转置。

In [ ]:
# ===== 5.5.6 例 5.18：非方图转置，越界补 0 =====
import cv2
import numpy as np

img = np.random.randint(0, 256, size=[4, 6], dtype=np.uint8)   # 故意 4 行 6 列
rows, cols = img.shape
mapx = np.zeros(img.shape, np.float32)
mapy = np.zeros(img.shape, np.float32)
for i in range(rows):
    for j in range(cols):
        mapx[i, j] = i                  # 原书：mapx.itemset((i,j), i)——行号当 x
        mapy[i, j] = j                  # 原书：mapy.itemset((i,j), j)——列号当 y
rst = cv2.remap(img, mapx, mapy, cv2.INTER_LINEAR)
print("rst=\n", rst)

### 例 5.19 · 图像 x 轴、y 轴互换

真实图像上的转置效果。

In [ ]:
# ===== 5.5.6 例 5.19：图像转置 =====
import cv2
import numpy as np

img = cv2.imread("picture1.jpg")        # 原书 lena.bmp
rows, cols = img.shape[:2]
mapx = np.zeros(img.shape[:2], np.float32)
mapy = np.zeros(img.shape[:2], np.float32)
for i in range(rows):
    for j in range(cols):
        mapx[i, j] = i
        mapy[i, j] = j
rst = cv2.remap(img, mapx, mapy, cv2.INTER_LINEAR)

cv2.imshow("original", img)
cv2.imshow("result", rst)
cv2.waitKey()
cv2.destroyAllWindows()

### 5.5.7 图像缩放

### 例 5.20 · remap 缩小图像（中心区域显示）

`remap` 的真正威力在浮点映射：用"取样步长为 2"的映射把图像缩小一半并固定在画布中心——目标图 (0.25·长度, 0.75·长度) 区间内 `mapx = 2*(j - cols*0.25) + 0.5`、`mapy = 2*(i - rows*0.25) + 0.5`，目标每前进 1 像素，取样位置前进 2 像素（起点加 0.5 让取样位置落在像素中心）；区域外的点统一取 (0, 0) 坐标的值，于是四周显示同一个像素值。

> **原书笔误**：原书条件写作 `0.25*cols < i < 0.75*cols and 0.25*rows < j < 0.75*rows`——i 是行号却与 cols 比，j 是列号却与 rows 比。lena 是 512×512 的方图，两组条件等价、结果碰巧正确；非方图上会选错区域。下面代码已按"行号与 rows 比、列号与 cols 比"更正，与例 5.8 的 pts2 是同一类问题。

In [ ]:
# ===== 5.5.7 例 5.20：中心区域显示缩小图 =====
import cv2
import numpy as np

img = cv2.imread("picture1.jpg")        # 原书 lena.bmp
rows, cols = img.shape[:2]
mapx = np.zeros(img.shape[:2], np.float32)
mapy = np.zeros(img.shape[:2], np.float32)
for i in range(rows):
    for j in range(cols):
        # 原书条件写作 0.25*cols < i < 0.75*cols and 0.25*rows < j < 0.75*rows
        if 0.25*rows < i < 0.75*rows and 0.25*cols < j < 0.75*cols:
                mapx[i, j] = 2*(j - cols*0.25) + 0.5    # 原书：mapx.itemset(...)
                mapy[i, j] = 2*(i - rows*0.25) + 0.5    # 原书：mapy.itemset(...)
        else:
                mapx[i, j] = 0
                mapy[i, j] = 0
rst = cv2.remap(img, mapx, mapy, cv2.INTER_LINEAR)

cv2.imshow("original", img)
cv2.imshow("result", rst)
cv2.waitKey()
cv2.destroyAllWindows()

## 5.6 极坐标变换（拓展）

补充内容（原书本章只在表 5-1 的 WARP_INVERSE_MAP 行提到极坐标变换，未展开，以下按 OpenCV 官方文档整理）：`cv2.warpPolar( src, dsize, center, maxRadius, flags )` 把以 center 为圆心的圆形区域展开成"角度 × 半径"的矩形——**dsize 的 width 是半径轴（列）、height 是角度轴（行）**。flags 用 `cv2.WARP_POLAR_LINEAR`（线性极坐标，值 0）或 `cv2.WARP_POLAR_LOG`（半对数极坐标，半径轴取对数刻度、中心附近分辨率更高，值 256），叠加 `cv2.WARP_INVERSE_MAP`（值 16）可做逆变换。dsize 两个值都 ≤0 时自动取（round(maxRadius), round(maxRadius·π)）。圆环展开成横带——极坐标图里"一条横带"就是原图里"一个圆环"，"一条竖带"就是"一条射线"；旧接口 `cv2.linearPolar`/`cv2.logPolar` 已废弃，新代码一律用 warpPolar。

In [ ]:
# ===== 5.6 拓展：圆环展开成横带 =====
import cv2
import numpy as np

img = np.zeros((201, 201), np.uint8)
cv2.circle(img, (100, 100), 50, 255, 6)     # 以 (100,100) 为圆心画半径 50 的白圆环

# dsize 传 (0,0)：自动取（round(100), round(100·π)）= (100, 314)
pol = cv2.warpPolar(img, (0, 0), (100, 100), 100, cv2.WARP_POLAR_LINEAR)
print(pol.shape)                            # (314, 100)：314 行是角度，100 列是半径

# 环上所有点的半径都是 50 → 聚成第 50 列附近的一条横向亮带
print(bool((pol[:, 48:53] > 200).all()))    # True

# 逆变换：flags 叠加 WARP_INVERSE_MAP，把极坐标图还原回直角坐标
back = cv2.warpPolar(pol, (201, 201), (100, 100), 100,
                     cv2.WARP_POLAR_LINEAR + cv2.WARP_INVERSE_MAP)